In [17]:
import pandas as pd
from collections import defaultdict
import json


In [31]:
# prepare app_metadata.json and app_categories.json
# app_metadata.json: item_id, app_package, app_name, item_id, price, avg_rating, num_reviews, app_categories
# app_categories: app_category_id, description, item_ids
app_categories_raw = pd.read_parquet('metadata/app_categories.parquet')
app_item_id_map_raw = pd.read_parquet('metadata/app_item_id_map.parquet')
app_metadata_raw = pd.read_parquet('metadata/app_metadata.parquet')

app_metadata = dict()
app_categories = dict()

# print(app_categories_raw.head())
# print(app_item_id_map_raw.head())
print(app_metadata_raw.tail())

app_package2item_id = dict()
item_id2app_package = dict()

for row in app_item_id_map_raw.itertuples():
    app_package2item_id[row.app_package] = row.item_id
    item_id2app_package[row.item_id] = row.app_package

category2item_ids = defaultdict(list)
category_description2app_category_id = dict()

for row in app_categories_raw.itertuples():
    category_description2app_category_id[row.category_description] = row.app_category_id

for row in app_metadata_raw.itertuples():
    if row.app_package not in app_package2item_id:
        raise KeyError(f'could not find item_id for app_package: {row.app_package}')
    item_id = app_package2item_id.get(row.app_package)
    category2item_ids[row.app_category].append(item_id)

    app_metadata[item_id] = {
        "item_id": item_id,
        "app_package": row.app_package,
        "app_name": row.app_name,
        "price": 0 if row.price.strip() in ["Install", "not found"] else float(row.price.split()[0][1:]),
        "num_reviews": int(row.num_reviews.replace(",", "")),
        "avg_rating": row.avg_rating,
        "category_ids": [category_description2app_category_id.get(row.app_category)] if row.app_category in category_description2app_category_id else [],
    }

for row in app_categories_raw.itertuples():
    app_categories[row.app_category_id] = {
        "app_category_id": row.app_category_id,
        "description": row.category_description,
        "item_ids": category2item_ids.get(row.category_description, [])
    }



with open("app_metadata.json", "w") as file:
    json.dump(app_metadata, file, indent=2)

with open("app_categories.json", "w") as file:
    json.dump(app_categories, file, indent=2)


                                 app_package  \
10168  com.unysyegor.effects.neon.magicslate   
10169               com.romancestories.novel   
10170              com.cornerdesk.gfx.nstate   
10171               com.picture.magic.imager   
10172                  cyberwolf.musicplayer   

                                    app_name          developer_name  \
10168             Magic Slate - Neon Effects               Unysyegor   
10169         Romance Stories-eBooks &Novels  Romance Stories Studio   
10170  PUBG NEW STATE : GFX Tool Pro + 90FPS         CornerDesk Inc.   
10171         Magic Photo Editor:Foto Repair            Zachary Holt   
10172             Music Player - MP3 & Audio           Elysium Group   

            app_category                                        description  \
10168      Entertainment  Magic Slate Neon Effects app contains on learn...   
10169  Books & Reference  Romance Stories - Romantic Story & Fantasy Nov...   
10170   Libraries & Demo  GFX Tool is a f

   user_id  sequence_length  \
0        1               84   
1        2                9   
2        3               25   
3        4                6   
4        5               35   

                                      train_sequence  \
0  [4380, 5456, 7967, 2300, 1250, 5562, 3622, 896...   
1          [3320, 9767, 1189, 8021, 3416, 212, 1764]   
2  [8086, 8734, 2608, 361, 8129, 6483, 8213, 4354...   
3                           [8742, 5756, 5560, 1673]   
4  [5481, 9838, 3375, 7533, 243, 7831, 5358, 5516...   

                                 validation_sequence  \
0  [4380, 5456, 7967, 2300, 1250, 5562, 3622, 896...   
1    [3320, 9767, 1189, 8021, 3416, 212, 1764, 2316]   
2  [8086, 8734, 2608, 361, 8129, 6483, 8213, 4354...   
3                     [8742, 5756, 5560, 1673, 5025]   
4  [5481, 9838, 3375, 7533, 243, 7831, 5358, 5516...   

                                       test_sequence  validation_target  \
0  [4380, 5456, 7967, 2300, 1250, 5562, 3622, 896...            